# 02.7 — Optimizers and learning rates

MATLAB hides the optimizer inside `trainingOptions('adam', ...)`. PyTorch hands you the pieces — optimizer object, learning-rate schedule, the update step — and expects you to assemble the loop. The upside: the pieces are simple, and total control is exactly what a custom pipeline like this one needs.

This notebook covers:

* `torch.optim`: constructing Adam / SGD(M), and what `.step()` actually does.
* The learning rate — the one hyperparameter that always matters — with a live comparison experiment.
* Schedulers: decaying the lr over training.
* Parameter groups: different settings for different parts of the model.
* A note on weight decay vs MATLAB's L2 (they are NOT the same thing).

**Prerequisites:** [02.5 autograd](02.5_autograd_basics.ipynb), [02.6 nn.Module](02.6_nn_module_vs_layergraph.ipynb).

## Section 1 — What MATLAB does

Declaratively, via `trainingOptions`:

```matlab
options = trainingOptions('adam', ...
    'InitialLearnRate', 1e-3, ...
    'LearnRateSchedule', 'piecewise', ...
    'LearnRateDropFactor', 0.9, ...
    'LearnRateDropPeriod', 30, ...
    'MiniBatchSize', 100, ...
    'MaxEpochs', 500);
net = trainNetwork(data, layers, options);
```

Or manually in a custom loop (the style `cgg_trainNetwork` uses):

```matlab
[net, avgGrad, avgSqGrad] = adamupdate(net, gradients, avgGrad, avgSqGrad, iter, lr);
```

Note that `adamupdate` makes YOU carry the optimizer's internal state (`avgGrad`, `avgSqGrad`) between iterations. PyTorch's optimizer objects hold that state internally — that's most of what they are.

## Section 2 — The Python concepts you need

### 2.1 — The optimizer object and the three-step loop

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# One training step — the eternal three-step rhythm (02.5 §2.3):
x, y = torch.randn(64, 10), torch.randn(64, 1)
optimizer.zero_grad()             # 1. clear old gradients
loss = loss_fn(model(x), y)
loss.backward()                    # 2. compute fresh ones
optimizer.step()                   # 3. apply:  p ← p - update(p.grad)
print(f"loss after one step: {loss.item():.4f}")

`optimizer.step()` is the whole `adamupdate` call — it reads every parameter's `.grad`, updates the internal running moments (MATLAB's `avgGrad`/`avgSqGrad`), and modifies the parameters in place. Passing `model.parameters()` at construction is how it knows *which* tensors to manage — which is why the registration rules from [02.6](02.6_nn_module_vs_layergraph.ipynb) matter so much.

**The optimizers you'll actually meet:**

| PyTorch | MATLAB analog | Notes |
|---|---|---|
| `torch.optim.Adam(params, lr=1e-3)` | `'adam'` / `adamupdate` | the default choice; per-parameter adaptive step sizes |
| `torch.optim.SGD(params, lr=..., momentum=0.9)` | `'sgdm'` / `sgdmupdate` | classic momentum SGD — this repo's `"SGDM"` config option maps here with momentum 0.9, matching `cgg_initializeOptimizerVariables.m` |
| `torch.optim.AdamW(params, ...)` | — | Adam with *decoupled* weight decay; see §2.5 |

### 2.2 — The learning rate, empirically

Rather than describe lr sensitivity, measure it. Same model, same data, same seed — only the optimizer setup differs:

In [ ]:
import matplotlib.pyplot as plt

def run(opt_name, epochs=150):
    torch.manual_seed(0)
    g = torch.Generator().manual_seed(1)
    x = torch.randn(256, 10, generator=g)
    w_true = torch.randn(10, 1, generator=g)
    y = x @ w_true + 0.1 * torch.randn(256, 1, generator=g)

    model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
    if opt_name == "SGD, lr too small":
        opt, sched = torch.optim.SGD(model.parameters(), lr=1e-4), None
    elif opt_name == "SGD+momentum":
        opt, sched = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9), None
    elif opt_name == "Adam":
        opt, sched = torch.optim.Adam(model.parameters(), lr=1e-2), None
    elif opt_name == "Adam + StepLR":
        opt = torch.optim.Adam(model.parameters(), lr=1e-2)
        sched = torch.optim.lr_scheduler.StepLR(opt, step_size=50, gamma=0.3)

    losses = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(x), y)
        loss.backward()
        opt.step()
        if sched is not None:
            sched.step()               # schedulers advance per EPOCH (here: per step)
        losses.append(loss.item())
    return losses

fig, ax = plt.subplots(figsize=(9, 4.5))
for name in ["SGD, lr too small", "SGD+momentum", "Adam", "Adam + StepLR"]:
    ax.semilogy(run(name), label=name, lw=2)
ax.set_xlabel("step"); ax.set_ylabel("MSE loss (log scale)")
ax.legend(); ax.set_title("Same model + data — only the optimizer setup differs")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Read the curves:

* **lr too small** — technically converging, practically frozen. The most common "my model doesn't train" cause.
* **momentum** rescues plain SGD by smoothing the descent direction.
* **Adam** self-tunes per-parameter step sizes — the reason it's the default for everything in this project.
* **StepLR** drops the lr ×0.3 every 50 steps: fast early progress, then finer steps once near a minimum — the log-scale kink at each drop is visible.

### 2.3 — Schedulers

A scheduler wraps an optimizer and adjusts its lr on a policy. The MATLAB `'piecewise'` schedule maps directly:

| MATLAB `trainingOptions` | PyTorch |
|---|---|
| `LearnRateSchedule='piecewise'`, `DropFactor=0.9`, `DropPeriod=30` | `StepLR(opt, step_size=30, gamma=0.9)` |
| (no analog) | `CosineAnnealingLR`, `ReduceLROnPlateau`, `OneCycleLR`, … |

The contract: call `scheduler.step()` once per epoch (after the epoch's training), and read the current value with `scheduler.get_last_lr()`. This repo's `base.yaml` carries the same three numbers as cfg fields — `initial_learning_rate: 0.001`, `learning_rate_decay: 0.9`, `learning_rate_epoch_drop: 30` — a direct port of the MATLAB piecewise schedule.

### 2.4 — Parameter groups: different rules for different parts

An optimizer can manage several **groups**, each with its own settings — the mechanism behind per-module learning rates:

In [ ]:
encoder = nn.Linear(8, 16)
classifier = nn.Linear(16, 3)

opt = torch.optim.Adam([
    {"params": encoder.parameters(),    "lr": 1e-4},   # gentle on the encoder
    {"params": classifier.parameters(), "lr": 1e-2},   # aggressive on the head
])
for i, group in enumerate(opt.param_groups):
    print(f"group {i}: lr={group['lr']}, {sum(p.numel() for p in group['params'])} params")

This is precisely how the production freeze/thaw curriculum works: `build_optimizer_with_module_groups` (`training/freezing.py`) creates one group per submodule (encoder / decoder / classifier), and the curriculum then scales each group's lr epoch-by-epoch — a per-module lr factor of 0 being a "soft freeze" that keeps optimizer momentum alive, mirroring MATLAB's `setLearnRateFactor`. (Deep dive planned as notebook 07.5.)

### 2.5 — Weight decay ≠ MATLAB's L2 (a warning shot)

Both MATLAB's `L2Regularization` and PyTorch's `weight_decay` flag penalize large weights, and with plain SGD they're equivalent. **With Adam they are not**: MATLAB adds `L2Factor * param` into the gradient *before* Adam's adaptive scaling; `torch.optim.AdamW` instead decays weights *outside* the adaptive step (decoupled decay — usually better). Same word, three different algorithms.

This repo needs bit-parity with MATLAB, so it implements the MATLAB-style grad-side L2 explicitly rather than trusting either flag — the full derivation is planned as notebook 06.8. Until then, one rule: **when porting, never assume `weight_decay=` reproduces `L2Regularization=`.**

## Section 3 — The `neural_data_decoding` implementation

The optimizer factory — where the MATLAB config string (`"ADAM"` / `"SGDM"`) becomes a PyTorch optimizer:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/freezing.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "SGDM_DEFAULT_MOMENTUM" in line and "=" in line)
for k in range(max(0, i - 4), min(i + 3, len(src))):
    print(f"{k + 1:4} | {src[k]}")
print("     ...")
i = next(j for j, line in enumerate(src) if line.startswith("def resolve_optimizer_factory"))
for k in range(i, min(i + 12, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `SGDM_DEFAULT_MOMENTUM = 0.9` with a comment citing the exact MATLAB source (`cgg_initializeOptimizerVariables.m`) — parity decisions leave a paper trail.
* The factory returns a *callable* that builds the optimizer — the functions-as-values pattern from [01.3 §2.5](../01_python_for_matlab_users/01.3_functions_and_lambdas.ipynb), so the same code path serves both plain and per-module-group construction.

## Section 4 — Hands-on exercises

### Exercise 4.1 — watch a schedule move

Build any optimizer with `lr=0.1` and a `StepLR(step_size=10, gamma=0.5)`. Step it 30 times and print the lr at steps 0, 9, 10, 19, 20, 29. Predict the six values first.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
opt = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.1)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)
for step in range(30):
    if step in (0, 9, 10, 19, 20, 29):
        print(f"step {step:2}: lr = {sched.get_last_lr()[0]:.4f}")
    opt.step()
    sched.step()

### Exercise 4.2 — break it on purpose

Re-run the §2.2 experiment with `Adam` at `lr=1.0`. What do the first 20 loss values do? (This is what divergence looks like — recognize it now, in a sandbox.)

In [ ]:
torch.manual_seed(0)
g = torch.Generator().manual_seed(1)
x = torch.randn(256, 10, generator=g); y = x @ torch.randn(10, 1, generator=g)
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
opt = torch.optim.Adam(model.parameters(), lr=1.0)
for step in range(20):
    opt.zero_grad(); loss = nn.functional.mse_loss(model(x), y); loss.backward(); opt.step()
    if step % 4 == 0:
        print(f"step {step:2}: loss = {loss.item():.2f}")

## Section 5 — Common errors

### Loss doesn't move at all

In rough order of likelihood: lr far too small (§2.2's flat curve); the parameters you care about aren't IN the optimizer (plain-list trap, [02.6 §2.2](02.6_nn_module_vs_layergraph.ipynb) — the optimizer only manages what it was handed at construction); everything is frozen (`requires_grad=False`); or `zero_grad()`/`backward()`/`step()` are out of order.

### Loss explodes / becomes `inf` or `nan`

lr too large (Exercise 4.2), or a genuinely broken loss. Halve the lr until stable; add gradient clipping (`torch.nn.utils.clip_grad_norm_` — this repo clips every step, `gradient_threshold: 100` in `base.yaml`). NaN specifically → [02.8](02.8_nan_handling.ipynb).

### `optimizer.step()` seems to do nothing after loading a model

You built the optimizer over the OLD model's parameters, then replaced the model. The optimizer holds references to the original tensors — reconstruct it after any model swap.

### Scheduler warning: `Detected call of lr_scheduler.step() before optimizer.step()`

Call order within an iteration is optimizer first, scheduler after (and typically scheduler once per *epoch*, not per batch).

### Adam trains fine; SGDM with the "same" settings doesn't

Not a bug — they need different learning rates (often 10–100× apart) and SGDM needs its momentum set. Never port an lr between optimizer families unchanged.

## Section 6 — Further reading

- [torch.optim docs](https://pytorch.org/docs/stable/optim.html) — every optimizer + scheduler, including the update formulas.
- [Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101) — the AdamW paper behind §2.5's warning.
- [`training/freezing.py`](../../src/neural_data_decoding/training/freezing.py) — the production factory + per-module groups.

Next up: **[02.8 NaN handling](02.8_nan_handling.ipynb)** — the last notebook of Module 02, and the one written for this dataset's 45,015 NaNs per trial.